# Notebook 08 - US Case Table
**Purpose** Merges our scraped and labeled FTC cases (from FTC directly) with Rafal's entries into our working US table

**Inputs**
- `data/ftc_labeled.csv` (339 cases)
- `data/ftc_rafal_cases.csv` (285 entries)

**Outputs**
- `data/us_cases.csv` (339 x 39)

**Decisions made**
- Rafal labels are primary and treated as base for the aggregate table, Rafal: 274 cases, ours: 65 cases, source recorded in attribute 'label source'
- Rafal had 9 URLs that accounted for 19 rows, these were collapsed to 9 rows, bringing his 285 entries to 275,
- 1 entry lacked 'statutory_topics' 2019 Facebook consent order, thus 274 entries
- - this case is accounted for in our scrape
- 5 superceded/redundant columns pruned, full lineage still available upstream

**Gaps/Shortcomings**
- 3 suspected URL misassignments in Rafal (Genica, 214 Technologies, AYLO)
- 13 label disagreements flagged via audit
- 17 cases to be ajudicated by hand, labeled for later

**Last run:** 2026-07-14 — 339 rows, agreements .989/.978/.985
- COPPA: 271/274 match = .989 agreement
- GLBA: 268/274 match = .978 agreement
- FCRA: 270/274 match = .985 agreement
- **worth noting** this is agreement between ftc scraped and Rafal, not necessarily ground truth

In [11]:
import pandas as pd
import ast

In [12]:
us = pd.read_csv("/Users/nic/Documents/MM2/data/ftc_labeled.csv")
raf = pd.read_csv("/Users/nic/Documents/MM2/data/ftc_rafal_cases.csv")

us["url"] = us["url"].str.rstrip("/")
raf["ftc_url"] = raf["ftc_url"].str.rstrip("/")
raf = raf.groupby("ftc_url", as_index=False).agg(lambda s: "; ".join(sorted(set(str(v) for v in s.dropna()))) if s.dtype == object else s.iloc[0])
raf = raf.add_prefix("rafal_")

merged = us.merge(raf, left_on="url", right_on="rafal_ftc_url", how="left")
print(merged.shape)
print("cases with Rafal labels:", merged["rafal_statutory_topics"].notna().sum())


(339, 42)
cases with Rafal labels: 274


In [13]:
merged["label_source"] = merged["rafal_statutory_topics"].notna().map({True: "rafal", False: "ours"})
merged["violation_labels_final"] = merged["rafal_statutory_topics"].fillna(merged["violation_labels"])
print(merged["label_source"].value_counts())

label_source
rafal    274
ours      65
Name: count, dtype: int64


In [14]:
for statute, tagkey in [("COPPA", "COPPA"), ("GLBA", "Gramm-Leach-Bliley"), ("FCRA", "Credit Reporting")]:
    both = merged[merged["label_source"] == "rafal"]
    his = both["rafal_statutory_topics"].str.contains(statute, na=False)
    ours = both["tags"].str.contains(tagkey, na=False)
    print(f"{statute}: agreement {(his == ours).mean():.3f} | disagreements: {both.loc[his != ours, 'case_name'].tolist()}")

##Pruning superceded attributes
drop_cols = ["statutes", "statutes_recovered", "penalty_usd", "rafal_ftc_url", "case_status",]
merged = merged.drop(columns=drop_cols)
print(f"dropped {len(drop_cols)} superceded/legacy columns, {merged.shape[1]} remain")

merged.to_csv("/Users/nic/Documents/MM2/data/us_cases.csv", index=False)
print(merged.shape)

COPPA: agreement 0.989 | disagreements: ['Microsoft Corporation, U.S. v.', 'Miniclip, In the Matter of', 'Retina-X Studios, LLC, In the Matter of']
GLBA: agreement 0.978 | disagreements: ['Equifax, Inc.', "Franklin's Budget Car Sales, Inc., also d/b/a Franklin Toyota/Scion, In the Matter of", 'Action Research Group, Inc., et al.', 'Goal Financial, LLC, In the Matter of', 'CEO Group, Inc. d/b/a Check Em Out, and Scott Joseph', 'Integrity Security & Investigation Services, Inc.']
FCRA: agreement 0.985 | disagreements: ['ITMedia Solutions LLC', 'BoostMyScore LLC', 'Sitesearch Corporation, Doing Business As LeapLab', 'Consumerinfo.com., Inc., d/b/a Experian Consumer Direct, Qspace, Inc., and Iplace Inc.']
dropped 5 superceded/legacy columns, 39 remain
(339, 39)


In [15]:
# Fix 1: normalize violation_labels_final format
# "ours" rows are Python list strings e.g. "['COPPA']" — strip to match rafal's plain format
def normalize_label(val):
    if pd.isna(val) or str(val).strip() == "[]":
        return pd.NA
    if str(val).startswith("["):
        try:
            items = ast.literal_eval(str(val))
            items = [("Section 5 Only" if i.strip() == "Section 5" else i.strip()) for i in items if i.strip()]
            return "; ".join(items) if items else pd.NA
        except:
            return val
    return val

merged["violation_labels_final"] = merged["violation_labels_final"].apply(normalize_label)
print("=== Label format after normalization ===")
print(merged["violation_labels_final"].value_counts().head(20))
print("\n--- ours rows only ---")
print(merged[merged["label_source"] == "ours"]["violation_labels_final"].value_counts())

# Fix 2: drop duplicate case rows (Upromise and Bonzi each have two FTC URLs; keep rafal row)
dup_urls = [
    "https://www.ftc.gov/legal-library/browse/cases-proceedings/102-3116-upromise-inc",
    "https://www.ftc.gov/legal-library/browse/cases-proceedings/bonzi-software-inc",
]
before = len(merged)
merged = merged[~merged["url"].isin(dup_urls)].reset_index(drop=True)
print(f"\n=== Dedup: {before} → {len(merged)} rows (dropped {before - len(merged)}) ===")
print(merged[merged["case_name"].isin(["Upromise, Inc.", "Bonzi Software, Inc."])][["case_name", "url", "label_source"]])

# Fix 3: mark the 3 empty-label rows as Unknown
empty_mask = merged["violation_labels_final"].isna() & (merged["label_source"] == "ours")
print(f"\n=== Empty-label rows: {empty_mask.sum()} ===")
print(merged[empty_mask][["case_name", "url"]])
merged.loc[empty_mask, "violation_labels_final"] = "Unknown"
print("\n--- label_source breakdown after all fixes ---")
print(merged["label_source"].value_counts())
print(f"\nFinal shape after fixes: {merged.shape}")
merged.to_csv("/Users/nic/Documents/MM2/data/us_cases.csv", index=False)
print("Saved after dedup/fixes.")

=== Label format after normalization ===
violation_labels_final
Section 5 Only                    184
FCRA                               55
COPPA                              46
GLBA                               22
Health Breach Notification          6
FCRA; GLBA                          6
Privacy Shield                      5
Other                               2
Privacy Shield; Section 5 Only      2
Other; Section 5 Only               2
TSR; GLBA                           2
CAN-SPAM                            1
FCRA; TSR                           1
TSR; FCRA                           1
TSR                                 1
Name: count, dtype: int64

--- ours rows only ---
violation_labels_final
FCRA                              18
Section 5 Only                    14
COPPA                             12
GLBA                               7
Privacy Shield                     5
Other                              2
Privacy Shield; Section 5 Only     2
Other; Section 5 Only             

In [16]:
##Previews the final table
import pandas as pd
us = pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv")
us.sample(3, random_state=1).T

,102,125,11
case_name,"VenPath, Inc., In the Matter of","Turn Inc., In the Matter of",Apitor
url,https://www.ftc.gov/legal-library/browse/cases...,https://www.ftc.gov/legal-library/browse/cases...,https://www.ftc.gov/legal-library/browse/cases...
date,2018-11-19,2017-04-21,2025-10-01
case_type,Administrative,Administrative,Federal
matter_number,1823144,1523099,NaN
summary,NaN,NaN,The FTC reached a settlement with Apitor Techn...
tags,Consumer Protection; international cooperation...,Consumer Protection; Office of Technology Rese...,Consumer Protection; Regional Offices; Bureau ...
long_title,"In the Matter of VenPath, Inc., a corporation.","In the Matter of Turn Inc., a corporation.","United States of America, Plaintiff, v. Apitor..."
press_release_urls,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/news-events/news/press-rel...
pdf_urls,https://www.ftc.gov/system/files/documents/cas...,https://www.ftc.gov/system/files/documents/cas...,https://www.ftc.gov/system/files/ftc_gov/pdf/A...


In [17]:
us = pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv")

disagreements = [
    'Microsoft Corporation',
    'Miniclip',
    'Retina-X Studios',
    'Equifax',
    "Franklin's Budget Car Sales",
    'Action Research Group',
    'Goal Financial',
    'CEO Group',
    'Integrity Security',
    'ITMedia Solutions',
    'BoostMyScore',
    'LeapLab',
    'Consumerinfo.com',
]

url_misassignments = [
    'Genica',
    '214 Technologies',
    'AYLO',
]

cols = ['case_name', 'tags', 'rafal_statutory_topics', 'violation_labels_final', 'url']

for name in disagreements + url_misassignments:
    row = us[us['case_name'].str.contains(name, case=False, na=False)]
    if len(row):
        print(f"\n{'='*60}")
        print(row[cols].to_string())
        print(f"TEXT: {str(row['press_release_text'].iloc[0])[:300]}")


                                   case_name                                                                                                                            tags rafal_statutory_topics violation_labels_final                                                                                               url
45            Microsoft Corporation, U.S. v.  Consumer Protection; Children's Online Privacy Protection Act (COPPA); Entertainment; Privacy and Security; Children's Privacy         Section 5 Only         Section 5 Only     https://www.ftc.gov/legal-library/browse/cases-proceedings/1923258-microsoft-corporation-us-v
313  Microsoft Corporation, In the Matter of                                                                        Consumer Protection; Privacy and Security; Data Security                    NaN  Other; Section 5 Only  https://www.ftc.gov/legal-library/browse/cases-proceedings/012-3240-microsoft-corporation-matter
TEXT: Microsoft will pay $20 million to settle F

In [18]:
us = pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv")

corrections = {
    "Microsoft Corporation, U.S. v.":          "COPPA",
    "Miniclip":                                 "COPPA",
    "Equifax, Inc.":                            "GLBA",
    "BoostMyScore":                             "Other",
}
mask315 = us["url"].str.contains("012-3240", na=False)
us.loc[mask315, "violation_labels_final"] = "Section 5 Only"
print(f"Case 315 fixed: {mask315.sum()} row(s)")

for name_frag, new_label in corrections.items():
    mask = us["case_name"].str.contains(name_frag, case=False, na=False)
    us.loc[mask, "violation_labels_final"] = new_label
    print(f"{name_frag}: {mask.sum()} row(s) updated → {new_label}")

# AYLO: normalize un-processed list string
aylo_mask = us["case_name"].str.contains("Aylo", case=False, na=False)
us.loc[aylo_mask, "violation_labels_final"] = "Section 5 Only"
print(f"Aylo: {aylo_mask.sum()} row(s) → Section 5 Only")

us.to_csv("/Users/nic/Documents/MM2/data/us_cases.csv", index=False)
print("Saved.")
print(us[us["case_name"].str.contains("Microsoft|Miniclip|Equifax, Inc|BoostMy|Aylo", case=False, na=False)][["case_name","violation_labels_final"]])

Case 315 fixed: 1 row(s)
Microsoft Corporation, U.S. v.: 1 row(s) updated → COPPA
Miniclip: 1 row(s) updated → COPPA
Equifax, Inc.: 1 row(s) updated → GLBA
BoostMyScore: 1 row(s) updated → Other
Aylo: 1 row(s) → Section 5 Only
Saved.
                                   case_name violation_labels_final
13                     Pornhub/Mindgeek/Aylo         Section 5 Only
45            Microsoft Corporation, U.S. v.                  COPPA
68                Miniclip, In the Matter of                  COPPA
75                          BoostMyScore LLC                  Other
97                             Equifax, Inc.                   GLBA
313  Microsoft Corporation, In the Matter of         Section 5 Only


In [19]:
pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv").shape

(337, 39)